In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
import re

#orginele dataset inlezen
df = pd.read_csv('Leesmateriaal_OW.csv')
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,Boek,3F,fictie,WOII,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Beschikbaar,NaN,Verhalen,Educatief Media,2025.0,Pdf,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO / NT2,Beschikbaar,48.8,Verhalen,Leeswereld,2005.0,Online,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO / NT2,Beschikbaar,49.2,Verhalen,NaN,2025.0,Online,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO / NT2,Beschikbaar,55.3,Verhalen,Uitgeverij Noord,2018.0,Online,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO / NT2,Beschikbaar,49.2,Verhalen,Uitgeverij Noord,NaN,Online,CAT-00005


In [28]:
#Analyse van de kolommen Themas_boek, trefwoorden en bron_url

def normaliseer_waarde(waarde):
    if pd.isna(waarde):
        return ""

    waarde = str(waarde).strip().casefold()

    # Spaties rondom puntkomma's negeren
    waarde = re.sub(r"\s*;\s*", ";", waarde)

    # Meerdere spaties vervangen door één spatie
    waarde = re.sub(r"\s+", " ", waarde)

    return waarde

# Waarden normaliseren
themas = df["Themas_boek"].map(normaliseer_waarde)
trefwoorden = df["trefwoorden"].map(normaliseer_waarde)
bron_url = df["bron_url"].map(normaliseer_waarde)

# Eerst bepalen of Themas_boek een URL bevat
is_url = themas.str.startswith(("http://", "https://"))

# Als Themas_boek een URL is, vergelijk met bron_url.
# Anders vergelijk met trefwoorden.
verwachte_waarde = trefwoorden.copy()
verwachte_waarde.loc[is_url] = bron_url.loc[is_url]

# Controleren welke rijen niet overeenkomen
verschilt = themas != verwachte_waarde

# Alleen afwijkende rijen weergeven
verschillen = df.loc[
    verschilt,
    [
        "Titel",
        "Themas_boek",
        "trefwoorden",
        "bron_url",
    ],
].copy()

# Laat zien waarmee Themas_boek per rij is vergeleken
verschillen["vergeleken_met"] = is_url.loc[verschillen.index].map(
    {
        True: "bron_url",
        False: "trefwoorden",
    }
)

if verschillen.empty:
    print("Alle waarden komen overeen.")
else:
    display(verschillen)

,Titel,Themas_boek,trefwoorden,bron_url,vergeleken_met
188,Zolang het tuin duurt 2,"zelfreflectie,doorzetten",zelfreflectie; doorzetten,NaN,trefwoorden
201,De kleine kust,"jongeren,vriendschap",jongeren; vriendschap,NaN,trefwoorden
231,Wat Joris niet wist,"doorzetten,liefde",doorzetten; liefde,NaN,trefwoorden
242,Sleutel zonder kust,liefde/samenleving,liefde; samenleving,NaN,trefwoorden
246,De brief van Jelle,geheimen|macht,geheimen; macht,NaN,trefwoorden
254,Een foto voor Sem,"veerkracht,oorlog",veerkracht; oorlog,NaN,trefwoorden
270,Het witte huis,reizen/oorlog,reizen; oorlog,NaN,trefwoorden
282,Grens in de haven,"doorzetten,stalking",doorzetten; stalking,NaN,trefwoorden


In [5]:
#Analyse van de kolommen genre en genre_kopie

# Waarden uit beide kolommen ophalen en normaliseren
genre = df["genre"].map(normaliseer_waarde)
genre_kopie = df["genre_kopie"].map(normaliseer_waarde)

# Alleen vergelijken als genre_kopie oorspronkelijk geen NaN is
heeft_genre_kopie = df["genre_kopie"].notna()

# Verschillen bepalen
verschilt_genre = (
    heeft_genre_kopie
    & (genre != genre_kopie)
)

# Alleen afwijkende rijen weergeven
genre_verschillen = df.loc[
    verschilt_genre,
    [
        "Titel",
        "genre",
        "genre_kopie",
    ],
].copy()

if genre_verschillen.empty:
    print("Alle ingevulde waarden in genre en genre_kopie komen overeen.")
else:
    display(genre_verschillen)

Alle ingevulde waarden in genre en genre_kopie komen overeen.


In [32]:
#Missing values in de "Auteur" kolom vullen met "Onbekend"
df["Auteur"] = df["Auteur"].fillna("Onbekend")
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Aangevuld (deels gesimuleerd),NaN,VERHALEN,Educatief Media,2025,PDF,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),48.8,verhalen,Leeswereld,2005,Digitaal,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,NaN,2025,Digitaal,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),55.3,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,Uitgeverij Noord,NaN,online,CAT-00005


In [33]:
lege_rijen = df[
    df["Type materiaal"].isna() |
    df["Type materiaal"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",NaN,3F+,NaN,NaN,Nederlands,200.0,617.0,NaN,MBO,NaN,54.0,verhalen,Leeswereld,NaN,Papier,CAT-00068


In [34]:
#in de "type materiaal" mist er 1 waarde waarvan we op hebben gezocht wat het is en dat invullen
df["Type materiaal"] = df["Type materiaal"].fillna("Boek")
df.iloc[67:68]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",Boek,3F+,NaN,NaN,Nederlands,200.0,617.0,NaN,MBO,NaN,54.0,verhalen,Leeswereld,NaN,Papier,CAT-00068


In [35]:
lege_niveaus = df[
    df["Niveau"].isna() |
    df["Niveau"].astype(str).str.strip().eq("")
]

lege_niveaus.head(10)

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,NaN,fictie,NaN,Nederlands,354.0,837.0,NaN,,gecontroleerd,66.5,verhalen,Leeswereld,NaN,papier,CAT-00038
38,Libello voor vrouwen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,NaN,fictie,NaN,Nederlands,153.0,248.0,NaN,NT2,Aangevuld (deels gesimuleerd),63.5,verhalen,Uitgeverij Noord,2013,Papier,CAT-00039
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),NaN,fictie,NaN,Nederlands,311.0,100.0,NaN,,Aangevuld (deels gesimuleerd),60.0,verhalen,Leeswereld,2099,Papier,CAT-00040
40,De Startweekkrant,Onbekend,Wekelijkse digitale krant met begrijpelijke ar...,digitale krant,NaN,fictie,NaN,Nederlands,244.0,386.0,NaN,NT2,Aangevuld (deels gesimuleerd),70.3,verhalen,Leeswereld,NaN,Digitaal,CAT-00041
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,NaN,informatief,NaN,Nederlands,2.0,7.0,https://nos.nl/artikel/2548636-tekorten-zorgpe...,,Aangevuld (deels gesimuleerd),43.5,verhalen,NaN,2013,PDF,CAT-00157
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",online artikel,NaN,informatief,NaN,Nederlands,1.0,14.0,https://nos.nl/artikel/2476626-nieuwe-generati...,NT2,Aangevuld (deels gesimuleerd),44.6,verhalen,Leeswereld,2023,online,CAT-00158


In [36]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
# Lege waarden in Niveau herkennen
lege_niveaus = (
    df["Niveau"].isna()
    | df["Niveau"].astype(str).str.strip().eq("")
)
# Alleen lege Niveau-waarden aanpassen bij genre fictie
voorwaarde = (
    lege_niveaus
    & df["genre"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("fictie")
)

df.loc[voorwaarde, "Niveau"] = "2F"
df.iloc[37:41]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,2F,fictie,NaN,Nederlands,354.0,837.0,NaN,,gecontroleerd,66.5,verhalen,Leeswereld,NaN,papier,CAT-00038
38,Libello voor vrouwen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,2F,fictie,NaN,Nederlands,153.0,248.0,NaN,NT2,Aangevuld (deels gesimuleerd),63.5,verhalen,Uitgeverij Noord,2013,Papier,CAT-00039
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),2F,fictie,NaN,Nederlands,311.0,100.0,NaN,,Aangevuld (deels gesimuleerd),60.0,verhalen,Leeswereld,2099,Papier,CAT-00040
40,De Startweekkrant,Onbekend,Wekelijkse digitale krant met begrijpelijke ar...,digitale krant,2F,fictie,NaN,Nederlands,244.0,386.0,NaN,NT2,Aangevuld (deels gesimuleerd),70.3,verhalen,Leeswereld,NaN,Digitaal,CAT-00041


In [37]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
# Lege waarden in Niveau herkennen
lege_niveaus = (
    df["Niveau"].isna()
    | df["Niveau"].astype(str).str.strip().eq("")
)
# Alleen lege Niveau-waarden aanpassen bij genre fictie
voorwaarde = (
    lege_niveaus
    & df["genre"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("informatief")
)

df.loc[voorwaarde, "Niveau"] = "3F"
df.iloc[156:158]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,3F,informatief,NaN,Nederlands,2.0,7.0,https://nos.nl/artikel/2548636-tekorten-zorgpe...,,Aangevuld (deels gesimuleerd),43.5,verhalen,NaN,2013,PDF,CAT-00157
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",online artikel,3F,informatief,NaN,Nederlands,1.0,14.0,https://nos.nl/artikel/2476626-nieuwe-generati...,NT2,Aangevuld (deels gesimuleerd),44.6,verhalen,Leeswereld,2023,online,CAT-00158


In [57]:
def verwerk_type_materiaal(df):
    masker = (
        df["Type materiaal"]
        .fillna("")
        .str.strip()
        .str.lower()
        .str.startswith("boek -")
    )

    # Haal de tekst na "Boek -" op en zet die in genre
    df.loc[masker, "genre"] = (
        df.loc[masker, "Type materiaal"]
        .str.split("-", n=1)
        .str[1]
        .str.strip()
        .str.capitalize()
    )

    # Maak bijvoorbeeld "Boek - thriller" gelijk aan "Boek"
    df.loc[masker, "Type materiaal"] = "Boek"

    return df

df = verwerk_type_materiaal(df)
df[["Titel", "Type materiaal", "genre"]].head(70)

,Titel,Type materiaal,genre
0,Het dagboek van Anne Frank,boek,fictie
1,Blauw Water,Boek,fictie
2,Boer vindt vrouw,Boek,fictie
3,Cadeau voor u!,Boek,fictie
4,De biecht,Boek,fictie
...,...,...,...
65,Moordonderzoek,Boek,fictie
66,Zonder spijt,Boek,Roman
67,Het geluid van de stilte,NaN,NaN
68,De zaak Ridouan T.,Boek,fictie


In [38]:
lege_rijen = df[
    df["doelgroep"].isna() |
    df["doelgroep"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,2F,fictie,NaN,Nederlands,354.0,837.0,NaN,,gecontroleerd,66.5,verhalen,Leeswereld,NaN,papier,CAT-00038
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),2F,fictie,NaN,Nederlands,311.0,100.0,NaN,,Aangevuld (deels gesimuleerd),60.0,verhalen,Leeswereld,2099,Papier,CAT-00040
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,3F,informatief,NaN,Nederlands,2.0,7.0,https://nos.nl/artikel/2548636-tekorten-zorgpe...,,Aangevuld (deels gesimuleerd),43.5,verhalen,NaN,2013,PDF,CAT-00157


In [ ]:
#Missing values in de "doelgroep" kolom vullen met "Onbekend"
#df["doelgroep"] = df["doelgroep"].fillna("Onbekend")

In [40]:
def maak_zoektekst(df):
    kolommen = ["Korte omschrijving", "genre", "trefwoorden"]

    test_df = df.copy()

    test_df["zoektekst"] = test_df[kolommen].apply(
        lambda rij: " ".join(
            rij.dropna().astype(str).str.strip()
        ),
        axis=1
    )

    return test_df


# De echte dataset blijft hierdoor ongewijzigd
dfNieuweKolom = maak_zoektekst(df)

dfNieuweKolom.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,zoektekst
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Aangevuld (deels gesimuleerd),NaN,VERHALEN,Educatief Media,2025,PDF,CAT-00001,Dagboek van een Joods meisje dat tijdens de Tw...
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),48.8,verhalen,Leeswereld,2005,Digitaal,CAT-00002,Spannende thriller waarin een vrouw wordt geco...
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,NaN,2025,Digitaal,CAT-00003,"Toegankelijk verhaal over liefde, relaties en ..."
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),55.3,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004,Bundel toegankelijke verhalen over verrassende...
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,Uitgeverij Noord,NaN,online,CAT-00005,Verhaal waarin verborgen geheimen en persoonli...


In [41]:
df.columns

Index(['Titel', 'Auteur', 'Korte omschrijving', 'Type materiaal', 'Niveau',
       'genre', 'trefwoorden', 'taal', 'aantal_paginas',
       'geschatte_leestijd_min', 'bron_url', 'doelgroep', 'gegevensstatus',
       'leescomplexiteit_score', 'interessecluster', 'uitgever',
       'publicatiejaar', 'beschikbaar_formaat', 'cataloguscode'],
      dtype='str')